# 07 · INT8 quantization

Quantize the ONNX model to INT8 to shrink it and speed it up for edge inference.
This measures the two effects that do not need hardware: size reduction and
per-class accuracy drop. The latency benefit needs TensorRT on the Jetson and is
not measured here, but the accuracy drop is hardware-independent and transfers.

## Calibration data

Static INT8 quantization needs representative images to learn the activation
ranges it maps to int8. These come from the train split only, never val/test, to
avoid leaking evaluation data into the deployed model. The full-frame transform
is used so the images match the model's fixed 720x960 input shape.

In [1]:
from pathlib import Path

import numpy as np
import torch
from onnxruntime.quantization import CalibrationDataReader

from cropweed_seg.dataset import PeanutDataset
from cropweed_seg.transforms import build_transforms

ROOT = Path.cwd().parent


class PeanutCalibrationReader(CalibrationDataReader):
    """Feeds train images to the quantizer to learn activation ranges.

    Calibration uses the TRAIN split only, never val/test, to avoid leaking
    evaluation data into the deployed model.
    """

    def __init__(self, root: Path, n_samples: int = 50):
        ds = PeanutDataset(root, "train", build_transforms("val"))
        # a representative subset is enough for calibration
        self.samples = [ds[i][0].unsqueeze(0).numpy() for i in range(n_samples)]
        self.idx = 0

    def get_next(self):
        if self.idx >= len(self.samples):
            return None
        sample = {"input": self.samples[self.idx]}
        self.idx += 1
        return sample


# quick check it yields the right shape
reader = PeanutCalibrationReader(ROOT, n_samples=3)
first = reader.get_next()
print("calibration sample shape:", first["input"].shape, first["input"].dtype)

calibration sample shape: (1, 3, 720, 960) float32


The reader yields full-frame 720x960 samples, matching the exported model's fixed
input. Train split for provenance, val transform for shape.

## Quantize and compare size

Static quantization with QDQ format and MinMax calibration over 50 train images.
The pre-processing warning is a suggestion, not an error; it can slightly improve
quantization quality and would be the first lever to try if the accuracy drop
were large.

In [2]:
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat

fp32_path = ROOT / "runs" / "focal_dice_s42" / "model.onnx"
int8_path = ROOT / "runs" / "focal_dice_s42" / "model_int8.onnx"

reader = PeanutCalibrationReader(ROOT, n_samples=50)

quantize_static(
    model_input=str(fp32_path),
    model_output=str(int8_path),
    calibration_data_reader=reader,
    quant_format=QuantFormat.QDQ,
    weight_type=QuantType.QInt8,
    activation_type=QuantType.QInt8,
)

fp32_mb = fp32_path.stat().st_size / 1e6
int8_mb = int8_path.stat().st_size / 1e6
print(f"fp32 model: {fp32_mb:.1f} MB")
print(f"int8 model: {int8_mb:.1f} MB")
print(f"size reduction: {fp32_mb / int8_mb:.2f}x")

fp32 model: 97.7 MB
int8 model: 24.6 MB
size reduction: 3.98x


97.7 MB to 24.6 MB, a 3.98x reduction, close to the 4x theoretical limit of going
from 4-byte floats to 1-byte ints. The model fits in a quarter of the space.

## Per-class accuracy drop

The number that matters: how much each class loses to quantization, measured on
test. weed is the concern, since minority classes can be more sensitive to
quantization. Reported per class, never aggregate alone, as throughout the
project.

In [3]:
import onnxruntime as ort

from cropweed_seg.metrics import ConfusionMatrixMetric

test_ds = PeanutDataset(ROOT, "test", build_transforms("test"))

def evaluate_onnx(model_path):
    session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
    metric = ConfusionMatrixMetric()
    for idx in range(len(test_ds)):
        image, mask = test_ds[idx]
        logits = session.run(["logits"], {"input": image.unsqueeze(0).numpy()})[0]
        metric.update(torch.from_numpy(logits).argmax(dim=1), mask.unsqueeze(0))
    return metric.compute()

fp32_iou = evaluate_onnx(fp32_path)
int8_iou = evaluate_onnx(int8_path)

print(f"{'class':<16} {'fp32':>8} {'int8':>8} {'drop':>8}")
for k in fp32_iou:
    drop = fp32_iou[k] - int8_iou[k]
    print(f"{k:<16} {fp32_iou[k]:>8.4f} {int8_iou[k]:>8.4f} {drop:>+8.4f}")

class                fp32     int8     drop
iou_background     0.9711   0.9710  +0.0001
iou_crop           0.9096   0.9095  +0.0001
iou_weed           0.6695   0.6696  -0.0001
miou               0.8500   0.8500  +0.0000


## Result

The per-class drop is essentially zero: background +0.0001, crop +0.0001, weed
-0.0001, mIoU unchanged. The hard class survives quantization without measurable
loss; weed's tiny negative drop is numerical noise, not an improvement. INT8 is
effectively free here in accuracy terms.

A clean result like this is partly method: calibration from representative train
images, no leak, a well-behaved U-Net. More sensitive models or poor calibration
degrade more.

This closes the hardware-free deployment work. What remains is the latency
benchmark, which needs TensorRT on Jetson: the ONNX to TensorRT conversion and
on-device timing. The accuracy and size results here transfer; the latency gain
is what the device adds.

## Calibration method comparison

MinMax is the default above. It maps the full activation range, so outliers
widen the range and coarsen the step for everything else. Entropy (KL) and
Percentile instead clip a tail to keep resolution where the mass is.

Hypothesis: if MinMax were penalizing weed via activation outliers, Entropy
should recover weed IoU disproportionately. The MinMax result already shows a
~zero drop, so the expected outcome is no meaningful difference, which would
confirm the activations are well-behaved and outlier-free. Run as an explicit,
documented close to the question.

Same calibration set across all methods (deterministic reader), same test split
to measure. Only the method varies.

In [4]:
from onnxruntime.quantization import CalibrationMethod

methods = {
    "minmax":  CalibrationMethod.MinMax,
    "entropy": CalibrationMethod.Entropy,
    "percentile": CalibrationMethod.Percentile,
}

method_paths = {}
for name, method in methods.items():
    out_path = ROOT / "runs" / "focal_dice_s42" / f"model_int8_{name}.onnx"

    reader = PeanutCalibrationReader(ROOT, n_samples=50)

    quantize_static(
        model_input=str(fp32_path),
        model_output=str(out_path),
        calibration_data_reader=reader,
        quant_format=QuantFormat.QDQ,
        weight_type=QuantType.QInt8,
        activation_type=QuantType.QInt8,
        calibrate_method=method,   # the one line that changes everything
    )
    method_paths[name] = out_path
    print(f"{name:<12} -> {out_path.name}")

minmax       -> model_int8_minmax.onnx


Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 117
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128


entropy      -> model_int8_entropy.onnx


Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 117
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)


percentile   -> model_int8_percentile.onnx


In [5]:
# reuse evaluate_onnx from above
fp32_iou = evaluate_onnx(fp32_path)
results = {name: evaluate_onnx(p) for name, p in method_paths.items()}

classes = list(fp32_iou.keys())
header = f"{'class':<16} {'fp32':>11} {'minmax':>11} {'entropy':>11} {'percentile':>11} {'Δ(ent-mm)':>11}"
print(header)
for k in classes:
    delta = results["entropy"][k] - results["minmax"][k]
    print(
        f"{k:<16} "
        f"{fp32_iou[k]:>11.4f} "
        f"{results['minmax'][k]:>11.4f} "
        f"{results['entropy'][k]:>11.4f} "
        f"{results['percentile'][k]:>11.4f} "
        f"{delta:>+11.4f}"
    )

class                   fp32      minmax     entropy  percentile   Δ(ent-mm)
iou_background        0.9711      0.9710      0.9710      0.9710     +0.0000
iou_crop              0.9096      0.9095      0.9095      0.9093     +0.0000
iou_weed              0.6695      0.6696      0.6696      0.6692     +0.0000
miou                  0.8500      0.8500      0.8500      0.8498     +0.0000
